## Topic 1: Classes, Objects, and self
```text
What is a class, really?

In Python, a class is itself an object — an instance of type. This is the first mental shift you need: everything in Python is an object, including classes themselves.
```

In [1]:
class Dog:
    pass

print(type(Dog))
print(type(Dog()))

<class 'type'>
<class '__main__.Dog'>


```text
So Dog is an instance of type, and any Dog() object is an instance of Dog. This chain matters later when we get to metaclasses.

Instance creation — what actually happens

When you write d = Dog(), two things happen behind the scenes:

Dog.__new__(Dog) is called → creates a raw, empty object in memory
Dog.__init__(d) is called on that object → initializes its state
```

In [2]:
class Dog:
    def __init__(self, name):
        self.name = name

d = Dog("Rex")

```text
Most people think __init__ "creates" the object. It doesn't — it just configures it after __new__ already made it.

self — the most misunderstood keyword that isn't a keyword

self is not a reserved word. It's just a convention. You could call it anything:
```

In [3]:
class Dog:
    def __init__(potato, name):
        potato.name = name

```text
When you call a method on an instance, Python automatically passes the instance as the first argument. self is just the name we give that first argument by convention. This is critical to understand because it explains:

Why staticmethod doesn't need self (no instance is passed)
Why calling Dog.some_method(d) and d.some_method() are equivalent
Why forgetting self in a method definition throws TypeError: missing 1 required positional argument
```

### Instance attributes vs class attributes — the core distinction

In [4]:
class Dog:
    species = "Canis familiaris"  # class attribute — shared by ALL instances

    def __init__(self, name):
        self.name = name          # instance attribute — unique per object

In [5]:
d1 = Dog("Rex")
d2 = Dog("Fido")

print(d1.species)  # Canis familiaris
print(d2.species)  # Canis familiaris
print(d1.name)     # Rex
print(d2.name)     # Fido

Canis familiaris
Canis familiaris
Rex
Fido


```text
Class attributes live on the class object. Instance attributes live on the individual object
```

In [6]:
d1.species = "Modified" # reassignment, creates new instance attribute
print(d1.species)  # Modified
print(d2.species)  # Canis familiaris (unchanged!)
print(Dog.species) # Canis familiaris (unchanged!)

Modified
Canis familiaris
Canis familiaris


```text
What happened? d1.species = "Modified" didn't change the class attribute — it created a new instance attribute on d1 that shadows the class attribute. d2 and Dog itself are untouched. This shadowing behavior is a core Python lookup rule:
```

In [7]:
print(d1.__dict__)
print(d2.__dict__)
print(Dog.__dict__)

{'name': 'Rex', 'species': 'Modified'}
{'name': 'Fido'}
{'__module__': '__main__', 'species': 'Canis familiaris', '__init__': <function Dog.__init__ at 0x000001532A836A20>, '__dict__': <attribute '__dict__' of 'Dog' objects>, '__weakref__': <attribute '__weakref__' of 'Dog' objects>, '__doc__': None}


In [29]:
class Animal:
    legs = 4
    def __init__(self,name):
        self.name = name
        

a1 = Animal('Dog')
a2 = Animal('Cat')
a3 = Animal('Kangaroo')
a3.legs = 2

print(a1)
print(a2)
print(a3)

print(a1.__dict__)
print(a2.__dict__)
print(a3.__dict__)

{'name': 'Dog'}
{'name': 'Cat'}
{'name': 'Kangaroo', 'legs': 2}


In [35]:
class Counter:
    count = 0
    def __init__(self):
        Counter.count += 1

a = Counter()
b = Counter()
c = Counter()

print(a.count, b.count, c.count, Counter.count)
print(a.__dict__)
print(b.__dict__)
print(c.__dict__)
print(Counter.__dict__)

3 3 3 3
{}
{}
{}
{'__module__': '__main__', 'count': 3, '__init__': <function Counter.__init__ at 0x000001532A93A5C0>, '__dict__': <attribute '__dict__' of 'Counter' objects>, '__weakref__': <attribute '__weakref__' of 'Counter' objects>, '__doc__': None}


In [36]:
class Counter:
    count = 0
    def __init__(self):
        Counter.count += 1
        self.count += 1

a = Counter()
b = Counter()
c = Counter()

print(a.count, b.count, c.count, Counter.count)
print(a.__dict__)
print(b.__dict__)
print(c.__dict__)
print(Counter.__dict__)

2 3 4 3
{'count': 2}
{'count': 3}
{'count': 4}
{'__module__': '__main__', 'count': 3, '__init__': <function Counter.__init__ at 0x000001532A9382C0>, '__dict__': <attribute '__dict__' of 'Counter' objects>, '__weakref__': <attribute '__weakref__' of 'Counter' objects>, '__doc__': None}


In [37]:
class Counter:
    count = 0
    def __init__(self):
        self.count += 1

a = Counter()
b = Counter()
c = Counter()

print(a.count, b.count, c.count, Counter.count)
print(a.__dict__)
print(b.__dict__)
print(c.__dict__)
print(Counter.__dict__)

1 1 1 0
{'count': 1}
{'count': 1}
{'count': 1}
{'__module__': '__main__', 'count': 0, '__init__': <function Counter.__init__ at 0x000001532A93ADE0>, '__dict__': <attribute '__dict__' of 'Counter' objects>, '__weakref__': <attribute '__weakref__' of 'Counter' objects>, '__doc__': None}


In [39]:
class Dog:
    tricks = []   # dangerous!

    def add_trick(self, trick):
        self.tricks.append(trick)  # this MUTATES the shared list, doesn't create a new one

d1 = Dog()
d2 = Dog()
d1.add_trick("sit")
print(d2.tricks)  # ['sit']  <- leaked into d2!

print(d1.__dict__)
print(d2.__dict__)
print(Dog.__dict__)

['sit']
{}
{}
{'__module__': '__main__', 'tricks': ['sit'], 'add_trick': <function Dog.add_trick at 0x000001532A954180>, '__dict__': <attribute '__dict__' of 'Dog' objects>, '__weakref__': <attribute '__weakref__' of 'Dog' objects>, '__doc__': None}


## Topic 2: Constructors and Dunder Methods

```text
__new__ is a static method (implicitly) responsible for creating and returning a new object. It receives the class (cls), not an instance.
__init__ receives the already-created object (self) and just sets up its state. It returns None implicitly — if you try to return anything else, Python raises a TypeError.
```

In [40]:
class Dog:
    def __new__(cls, *args, **kwargs):
        print('1. __new__ called - creating instance')
        return super().__new__(cls)
    def __init__(self, name):
        print('2. __init__ called - initializing instance')
        self.name = name
        
d = Dog('Rex')
        

1. __new__ called - creating instance
2. __init__ called - initializing instance


```text
if __new__ doesn't return an instance of cls, __init__ is never called at all.
```

In [41]:
class Dog:
    def __new__(cls, *args, **kwargs):
        return "not a dog instance"  # returning something else entirely

    def __init__(self, name):
        print("This will NEVER print")
        self.name = name

d = Dog("Rex")
print(d)          # "not a dog instance"
print(type(d))    # <class 'str'>

not a dog instance
<class 'str'>


In [43]:
class A:
    def __new__(cls):
        print("new")
        return super().__new__(cls)

    def __init__(self):
        print("init")

class B(A):
    pass

b = B()

new
<class '__main__.B'>
init


In [44]:
class A:
    def __new__(cls):
        print("new, cls =", cls)
        instance = super().__new__(cls)
        print("type of created instance:", type(instance))
        return instance

    def __init__(self):
        print("init")

class B(A):
    pass

b = B()

new, cls = <class '__main__.B'>
type of created instance: <class '__main__.B'>
init


In [45]:
class A:
    def __new__(cls):
        print("A.__new__, cls =", cls.__name__)
        return super().__new__(cls)

    def __init__(self):
        print("A.__init__")

class B(A):
    def __init__(self):
        print("B.__init__")
        super().__init__()

b = B()
print(type(b))

A.__new__, cls = B
B.__init__
A.__init__
<class '__main__.B'>


In [51]:
class X:
    def __new__(cls):
        print('X.__new__ , cls = ', cls)
        return super().__new__(cls)
        
    def __init__(self):
        print('X.__init__, self = ', self)

class Y(X):
    def __new__(cls):
        print('Y.__new__ , cls = ', cls)
        return super().__new__(cls)
        
    def __init__(self):
        print('Y.__init__, self = ', self)

y = Y()

Y.__new__ , cls =  <class '__main__.Y'>
X.__new__ , cls =  <class '__main__.Y'>
Y.__init__, self =  <__main__.Y object at 0x000001532A97C1D0>


## 3. self

In [71]:
class Chaicup:
    size = 150 #ml
    def describe(self):
        return f"A {self.size}ml chai cup"

cup_one = Chaicup()
cup_two = Chaicup()

print(Chaicup.describe(cup_one))
print(Chaicup.__dict__)
    

A 150ml chai cup
{'__module__': '__main__', 'size': 150, 'describe': <function Chaicup.describe at 0x000001532A9C1A80>, '__dict__': <attribute '__dict__' of 'Chaicup' objects>, '__weakref__': <attribute '__weakref__' of 'Chaicup' objects>, '__doc__': None}


### Excercise
```text
Smart Home Device Tracker
You’re building a Smart Home Device Tracker to monitor the status of electronic devices.


Your Tasks:

Create a class called SmartDevice.

Add a class attribute brand = "HomeTech" (default manufacturer).

In the constructor, accept:

device_name: Name of the smart device.

power_status: True (ON) or False (OFF).

Shadow the class attribute brand by assigning self.brand = "CustomBrand" (to simulate user modification).

Define a method get_status():

Returns a string like: "AC is ON - CustomBrand" or "Fan is OFF - CustomBrand" based on power_status.
```

In [ ]:
class SmartDevice:
    brand = "HomeTech"
    
    def __init__(self, device_name:str, power_status:bool):
        self.device_name = device_name
        self.power_status = power_status
        self.brand = "CustomBrand"
        
    def get_status(self)->str:
        status = "ON" if self.power_status else "OFF"
        return f"{self.device_name} is {status} - {self.brand}"

ac = 